In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
df = pd.read_csv("/content/Student_performance_data _.csv")
df.to_sql('students', conn, index=False, if_exists='replace')
print("Database ready!")
print(f"Total rows: {len(df)}")
def run_query(query):
    return pd.read_sql_query(query, conn)

Database ready!
Total rows: 2392


In [14]:
query1 = """
SELECT StudentID, Age, GPA, GradeClass
FROM students
WHERE GPA >= 3.5
ORDER BY GPA DESC
LIMIT 5
"""
print("Top 5 High GPA Students:")
print(run_query(query1))

query2 = """
SELECT StudentID, StudyTimeWeekly, Absences, GPA
FROM students
WHERE Absences > 20 AND GPA < 1.5
"""
print("\nAt-Risk Students:")
print(run_query(query2))

Top 5 High GPA Students:
   StudentID  Age  GPA  GradeClass
0       1045   18  4.0         0.0
1       1443   15  4.0         0.0
2       2279   15  4.0         0.0
3       2706   18  4.0         0.0
4       2920   15  4.0         0.0

At-Risk Students:
     StudentID  StudyTimeWeekly  Absences       GPA
0         1003         4.210570        26  0.112602
1         1008        15.424496        22  1.360143
2         1019        16.254658        29  0.469553
3         1022        15.323142        25  0.346894
4         1023        18.648880        29  0.312546
..         ...              ...       ...       ...
634       3357         7.262652        22  0.766257
635       3371         2.912575        25  0.569420
636       3374        19.088954        25  0.825020
637       3376        18.925290        24  1.164539
638       3379        12.905555        26  0.709353

[639 rows x 4 columns]


In [15]:
query1 = """
SELECT
    COUNT(*) as total_students,
    AVG(GPA) as avg_gpa,
    MAX(GPA) as max_gpa,
    MIN(GPA) as min_gpa
FROM students
"""
print("Overall Statistics:")
print(run_query(query1))

query2 = """
SELECT
    Gender,
    COUNT(*) as student_count,
    AVG(GPA) as avg_gpa,
    AVG(Absences) as avg_absences
FROM students
GROUP BY Gender
"""
print("\nGender-wise Summary:")
print(run_query(query2))

query3 = """
SELECT
    GradeClass,
    COUNT(*) as count,
    AVG(StudyTimeWeekly) as avg_study_time
FROM students
GROUP BY GradeClass
HAVING COUNT(*) > 100
ORDER BY GradeClass
"""
print("\nGrades with more than 100 students:")
print(run_query(query3))

Overall Statistics:
   total_students   avg_gpa  max_gpa  min_gpa
0            2392  1.906186      4.0      0.0

Gender-wise Summary:
   Gender  student_count   avg_gpa  avg_absences
0       0           1170  1.918679     14.355556
1       1           1222  1.894225     14.719313

Grades with more than 100 students:
   GradeClass  count  avg_study_time
0         0.0    107       11.854926
1         1.0    269       11.122335
2         2.0    391       10.106404
3         3.0    414        9.757963
4         4.0   1211        9.184822


In [16]:
query1 = """
SELECT
    StudentID,
    GPA,
    CASE
        WHEN GPA >= 3.5 THEN 'Excellent'
        WHEN GPA >= 2.5 THEN 'Good'
        WHEN GPA >= 1.5 THEN 'Average'
        ELSE 'Poor'
    END as Performance
FROM students
LIMIT 10
"""
print("Performance Categories:")
print(run_query(query1))

query2 = """
SELECT
    CASE
        WHEN Absences <= 5 THEN 'Low'
        WHEN Absences <= 15 THEN 'Medium'
        ELSE 'High'
    END as AbsenceLevel,
    COUNT(*) as count,
    AVG(GPA) as avg_gpa
FROM students
GROUP BY AbsenceLevel
"""
print("\nAbsence Level Analysis:")
print(run_query(query2))

Performance Categories:
   StudentID       GPA Performance
0       1001  2.929196        Good
1       1002  3.042915        Good
2       1003  0.112602        Poor
3       1004  2.054218     Average
4       1005  1.288061        Poor
5       1006  3.084184        Good
6       1007  2.748237        Good
7       1008  1.360143        Poor
8       1009  2.896819        Good
9       1010  3.573474   Excellent

Absence Level Analysis:
  AbsenceLevel  count   avg_gpa
0         High   1120  1.128477
1          Low    452  3.095303
2       Medium    820  2.312959


In [17]:
import pandas as pd

tutoring_data = pd.DataFrame({
    'TutoringCenter': [1, 0],
    'CenterName': ['Bright Minds Tutoring', 'No Tutoring'],
    'MonthlyFee': [2000, 0]
})
tutoring_data.to_sql('tutoring_centers', conn, index=False, if_exists='replace')

print("Tutoring Centers Table:")
print(tutoring_data)

query1 = """
SELECT
    s.StudentID,
    s.GPA,
    t.CenterName,
    t.MonthlyFee
FROM students s
INNER JOIN tutoring_centers t
ON s.Tutoring = t.TutoringCenter
LIMIT 5
"""
print("\nINNER JOIN Result:")
print(run_query(query1))

query2 = """
SELECT
    t.CenterName,
    COUNT(s.StudentID) as student_count,
    AVG(s.GPA) as avg_gpa
FROM students s
INNER JOIN tutoring_centers t
ON s.Tutoring = t.TutoringCenter
GROUP BY t.CenterName
"""
print("\nGPA by Tutoring Status:")
print(run_query(query2))

Tutoring Centers Table:
   TutoringCenter             CenterName  MonthlyFee
0               1  Bright Minds Tutoring        2000
1               0            No Tutoring           0

INNER JOIN Result:
   StudentID       GPA             CenterName  MonthlyFee
0       1001  2.929196  Bright Minds Tutoring        2000
1       1002  3.042915            No Tutoring           0
2       1003  0.112602            No Tutoring           0
3       1004  2.054218            No Tutoring           0
4       1005  1.288061  Bright Minds Tutoring        2000

GPA by Tutoring Status:
              CenterName  student_count   avg_gpa
0  Bright Minds Tutoring            721  2.108325
1            No Tutoring           1671  1.818968


In [18]:
query1 = """
SELECT
    StudentID,
    GPA,
    StudyTimeWeekly
FROM students
WHERE Tutoring = 1
  AND ParentalSupport >= 3
ORDER BY GPA DESC
LIMIT 5
"""

print("Query 1 Results:")
print(run_query(query1))

query2 = """
SELECT
    CASE
        WHEN StudyTimeWeekly > 15 THEN 'High Study'
        WHEN StudyTimeWeekly >= 10 THEN 'Medium Study'
        ELSE 'Low Study'
    END AS category,
    COUNT(*) AS count,
    AVG(GPA) AS avg_gpa,
    AVG(Absences) AS avg_absences
FROM students
GROUP BY category
ORDER BY category
"""

print("\nQuery 2 Results:")
print(run_query(query2))

query3 = """
SELECT
    t.CenterName,
    AVG(s.GPA) AS avg_gpa
FROM students s
INNER JOIN tutoring_centers t
    ON s.Tutoring = t.TutoringCenter
GROUP BY t.CenterName
ORDER BY t.TutoringCenter DESC
"""

print("\nQuery 3 Results:")
print(run_query(query3))

query4_count = """
SELECT COUNT(*) AS at_risk_count
FROM students
WHERE Absences > 15
  AND GPA < 1.5
  AND Tutoring = 0
"""

count_result = run_query(query4_count)

query4_students = """
SELECT StudentID
FROM students
WHERE Absences > 15
  AND GPA < 1.5
  AND Tutoring = 0
LIMIT 5
"""

students_result = run_query(query4_students)

print("\nQuery 4 Results:")
print(
    f"Count of at-risk students without tutoring: "
    f"{count_result.iloc[0]['at_risk_count']}"
)
print(
    f"First 5: "
    f"{students_result['StudentID'].tolist()}"
)

Query 1 Results:
   StudentID       GPA  StudyTimeWeekly
0       1045  4.000000        18.921512
1       2279  4.000000        18.899696
2       2706  4.000000         8.858282
3       3029  4.000000        18.656924
4       2261  3.984674         9.001905

Query 2 Results:
       category  count   avg_gpa  avg_absences
0    High Study    535  2.105937     14.908411
1     Low Study   1238  1.773450     14.465267
2  Medium Study    619  1.999016     14.376414

Query 3 Results:
              CenterName   avg_gpa
0  Bright Minds Tutoring  2.108325
1            No Tutoring  1.818968

Query 4 Results:
Count of at-risk students without tutoring: 624
First 5: [1003, 1019, 1022, 1033, 1034]
